# PyMC (Bayesian)

**Domain:** Data Analysis & Research  ·  *recommended addition*  ·  **runnable:** yes

A refresher on **PyMC** — a Python library for **Bayesian statistical modeling** and
probabilistic programming. You write down a generative model, hand it your data, and
PyMC's MCMC samplers (NUTS by default) return the *full posterior distribution* over
your parameters — not just point estimates.

## 1. What & Why

**What it is.** PyMC is a *probabilistic programming language* (PPL) embedded in Python.
You declare random variables (priors), connect them to data through a likelihood, and PyMC
draws samples from the resulting posterior with gradient-based MCMC. Under the hood it
compiles your model graph with **PyTensor** (autodiff + C/Numba backends) and samples with
the **No-U-Turn Sampler (NUTS)**.

**The problem it solves.** Classical (frequentist) tools give you a point estimate and a
confidence interval whose interpretation is famously slippery. Bayesian inference instead
returns a probability distribution over the unknowns, so you can answer questions directly:
*"Given the data, what's the probability this coefficient is positive?"* or *"What's the 89%
credible interval for the treatment effect?"* It naturally handles small samples, hierarchical
/ multilevel structure, and full uncertainty propagation into predictions.

**Reach for it when:**
- You want **calibrated uncertainty**, not just a best-fit number.
- You have **hierarchical / grouped** data (students in schools, measurements per sensor) and want partial pooling.
- You have **prior knowledge** worth encoding, or **small / messy** data where regularization matters.
- You need a **custom likelihood** that no off-the-shelf `statsmodels`/`sklearn` model provides.

**Skip it when** a fast closed-form or maximum-likelihood fit is enough, the dataset is huge
and you only need a point prediction, or latency is critical — MCMC is *much* slower than an
OLS or gradient-boosted fit. See [`statsmodels`](statsmodels.ipynb) for the frequentist side.

## 2. Mental Model

**Bayesian inference is bookkeeping for uncertainty.** You start with what you believe before
seeing data (the **prior**), you state how data is generated given the parameters (the
**likelihood**), and Bayes' rule combines them into what you believe *after* (the
**posterior**):

```
posterior  ∝  likelihood × prior
p(θ | data) ∝  p(data | θ) × p(θ)
```

The posterior is rarely solvable on paper, so PyMC **draws samples** from it instead. Think of
the sampler as a hiker wandering the parameter landscape, spending time in each region in
proportion to its posterior probability. The cloud of samples it leaves behind *is* your
answer — every quantity you care about (means, intervals, probabilities, predictions) is just
a summary of that cloud.

```
   prior beliefs ──┐
                   ├──►  PyMC model graph  ──►  NUTS sampler  ──►  posterior samples
   data + likelihood┘                                              (an InferenceData cloud)
                                                                        │
                          summarize ◄───────────────────────────────────┘
                       (mean, 89% interval, P(θ>0), posterior-predictive)
```

The `with pm.Model() as m:` block is just a context manager that registers every random
variable you create into the same graph.

## 3. Key Concepts

- **Prior** — `pm.Normal("mu", 0, 10)`. Your belief about a parameter before data. Weakly-informative priors (wide but not flat) regularize and speed up sampling.
- **Likelihood / observed variable** — a distribution with `observed=data`. This is how the data constrains the parameters.
- **Posterior** — the updated distribution after combining prior × likelihood. PyMC returns *samples* from it, packed into an `InferenceData` object.
- **NUTS** — the default sampler, an adaptive Hamiltonian Monte Carlo variant that uses gradients to explore efficiently. Works on continuous parameters; PyMC auto-assigns other samplers (e.g. Metropolis) to discrete ones.
- **Chains, tuning, draws** — `pm.sample(draws, tune, chains)`. Each chain is an independent run; `tune` steps adapt the sampler and are discarded; `draws` are the kept samples per chain.
- **`InferenceData`** — the xarray-backed container (an [ArviZ](https://python.arviz.org) structure) holding `posterior`, `sample_stats`, `observed_data`, etc.
- **R-hat (`r_hat`)** — convergence diagnostic comparing within- vs between-chain variance. **Want ≈ 1.00**; > 1.01 means chains disagree → don't trust the results.
- **ESS (`ess_bulk` / `ess_tail`)** — effective sample size after accounting for autocorrelation. Low ESS = your samples are correlated and you have less information than the raw count suggests.
- **Divergences** — NUTS warnings that it hit pathological geometry; more than a handful means the posterior is mis-specified or needs reparameterization.
- **Posterior predictive** — `pm.sample_posterior_predictive(...)`. Simulates new data from the fitted model; used for predictions and for checking model fit.
- **Credible interval** — e.g. the 89% interval genuinely contains the parameter with 89% probability given the model and data (unlike a frequentist confidence interval).

## 4. Setup

PyMC needs `pymc` (which pulls in `pytensor` for the compute graph) and `arviz` for
diagnostics and plotting. A C compiler or Numba speeds up sampling but the pure-Python
fallback works for small models.

```bash
pip install pymc arviz       # or:  conda install -c conda-forge pymc
```

The cell below installs into the running kernel if PyMC is missing, then imports everything.

In [ ]:
# Install on demand (no-op if already present), then import.
try:
    import pymc as pm
except ImportError:
    %pip install -q pymc arviz
    import pymc as pm

import numpy as np
import arviz as az

print("PyMC :", pm.__version__)
print("ArviZ:", az.__version__)

## 5. Worked Examples

### Example 1 — Estimating a mean with full uncertainty

The "hello world" of Bayesian inference: we have 50 noisy measurements and want the
underlying mean and spread. Instead of a single `np.mean`, we get a *distribution* over
plausible means.

In [ ]:
# Synthetic data: true mean 5.0, true sd 2.0
rng = np.random.default_rng(42)
y = rng.normal(5.0, 2.0, size=50)

with pm.Model() as mean_model:
    # Priors: weakly-informative, centered far from the truth on purpose.
    mu = pm.Normal("mu", mu=0.0, sigma=10.0)
    sigma = pm.HalfNormal("sigma", sigma=5.0)   # sd must be positive

    # Likelihood: the data is Normal(mu, sigma)
    pm.Normal("obs", mu=mu, sigma=sigma, observed=y)

    # Draw from the posterior (cores=1 keeps it portable inside notebooks)
    idata = pm.sample(1000, tune=1000, chains=4, cores=1,
                      random_seed=42, progressbar=False)

az.summary(idata, var_names=["mu", "sigma"], ci_prob=0.89)

Read the table: `mean` is the posterior mean (our best point estimate), `sd` is the
posterior uncertainty, and the `hdi`/`eti` columns give the 89% credible interval. The data
pulls the posterior to ≈ 5.0 and ≈ 2.0 despite the prior sitting at 0 — and crucially
`r_hat` ≈ 1.00 with healthy `ess`, so the chains converged. Let's turn the posterior cloud
into a direct probability statement.

In [ ]:
# Posterior samples are just an array — ask any question you like.
mu_samples = idata.posterior["mu"].values.ravel()

print(f"posterior mean of mu      : {mu_samples.mean():.3f}")
print(f"89% credible interval     : "
      f"[{np.percentile(mu_samples, 5.5):.3f}, {np.percentile(mu_samples, 94.5):.3f}]")
print(f"P(mu > 4.5 | data)        : {(mu_samples > 4.5).mean():.3f}")
print(f"P(mu > 5.5 | data)        : {(mu_samples > 5.5).mean():.3f}")

### Example 2 — Bayesian linear regression

The same recipe scales to a model with predictors. We recover a known slope and intercept,
then use the posterior predictive distribution to make uncertainty-aware predictions.

In [ ]:
# Synthetic data: y = 1.0 + 2.5*x + noise
rng = np.random.default_rng(0)
n = 80
x = rng.normal(size=n)
true_intercept, true_slope, true_noise = 1.0, 2.5, 0.5
y = true_intercept + true_slope * x + rng.normal(0, true_noise, size=n)

with pm.Model() as linreg:
    intercept = pm.Normal("intercept", 0.0, 5.0)
    slope = pm.Normal("slope", 0.0, 5.0)
    noise = pm.HalfNormal("noise", 2.0)

    mu = intercept + slope * x                      # deterministic linear predictor
    pm.Normal("y", mu=mu, sigma=noise, observed=y)  # likelihood

    idata_lr = pm.sample(1000, tune=1000, chains=4, cores=1,
                        random_seed=1, progressbar=False)
    # Simulate y for the observed x's, stored back into idata_lr
    pm.sample_posterior_predictive(idata_lr, extend_inferencedata=True,
                                   random_seed=1, progressbar=False)

az.summary(idata_lr, var_names=["intercept", "slope", "noise"], ci_prob=0.89)

In [ ]:
# The posterior recovers the true parameters (1.0, 2.5, 0.5) with quantified uncertainty.
slope_samples = idata_lr.posterior["slope"].values.ravel()
print(f"P(slope > 2.0 | data)  : {(slope_samples > 2.0).mean():.3f}")

# Posterior-predictive mean and 89% interval at each observed point.
ppc = idata_lr.posterior_predictive["y"]              # dims: chain, draw, obs
pred_mean = ppc.mean(dim=["chain", "draw"]).values
lo, hi = np.percentile(ppc.values.reshape(-1, n), [5.5, 94.5], axis=0)

covered = ((y >= lo) & (y <= hi)).mean()
print(f"share of points inside 89% predictive band: {covered:.2f}")
print(f"first 3 predictions (mean [lo, hi]):")
for i in range(3):
    print(f"  x={x[i]:+.2f} -> {pred_mean[i]:+.2f}  [{lo[i]:+.2f}, {hi[i]:+.2f}]")

## 6. Gotchas & Pitfalls

- **Always check `r_hat` and divergences before reading results.** A pretty `summary` table
  from chains that didn't converge is worse than no answer. `r_hat > 1.01`, low `ess`, or many
  *divergences* mean the samples don't represent the posterior — fix the model, don't report it.
- **Divergences usually signal geometry, not bugs.** The classic fix for hierarchical models is
  a **non-centered reparameterization** (sample a standard normal and scale it) rather than
  cranking `target_accept` forever. Bumping `pm.sample(..., target_accept=0.95)` helps mild cases.
- **`cores > 1` can hang or error inside notebooks** on Windows/macOS due to multiprocessing
  spawn semantics. Use `cores=1` (sequential chains) in notebooks, or guard scripts with
  `if __name__ == "__main__":`.
- **Flat / improper priors are a trap.** "Uninformative" priors slow sampling and can make the
  posterior improper. Prefer *weakly-informative* priors (e.g. `Normal(0, 10)`), and run a
  **prior predictive check** (`pm.sample_prior_predictive`) to confirm they imply sane data.
- **Standardize your predictors.** Wildly different scales hurt NUTS step-size adaptation; center
  and scale continuous inputs for faster, healthier sampling.
- **`observed=` is what makes a variable data.** Forgetting it (or passing the wrong array shape)
  silently changes the model. Double-check shapes with `pm.model_to_graphviz(model)`.
- **The first run is slow** because PyTensor compiles the graph; subsequent samples reuse it.
  Don't mistake compile time for sampling time.
- **`InferenceData` is xarray, not pandas.** Index with `.posterior["name"]` and reduce with
  `.mean(dim=[...])`; `.values.ravel()` gets you a flat NumPy array when you want raw samples.

## 7. When to Use vs Alternatives

| Tool | Paradigm | Use it when | Trade-off |
|------|----------|-------------|-----------|
| **PyMC** | Bayesian PPL (Python) | Custom models, hierarchy, full uncertainty, Pythonic API | Slower than MLE; MCMC tuning has a learning curve |
| **Stan** (cmdstanpy) | Bayesian PPL (own language) | Max performance, mature ecosystem, reproducible papers | Separate modeling language; steeper setup |
| **NumPyro** | Bayesian PPL (JAX) | Same models, GPU/TPU, huge speedups via JIT | JAX idioms; smaller high-level helper layer |
| **statsmodels** | Frequentist | Standard OLS/GLM/time-series, p-values, speed | Point estimates + CIs only; no custom generative models |
| **scikit-learn** | ML / optimization | Prediction at scale, pipelines, cross-validation | No principled parameter uncertainty |
| **Bambi** | Formula API *on* PyMC | Quick GLMM-style models via `y ~ x + (1|group)` | Less control than hand-built PyMC models |

**Rule of thumb:** if the question is *"what's the effect and how sure am I?"* with structure or
priors that matter, use PyMC. If it's *"give me an accurate prediction fast on lots of data,"*
reach for scikit-learn or gradient boosting. For frequentist regression with classical
inference, [`statsmodels`](statsmodels.ipynb) is simpler and faster. Choose **NumPyro/Stan**
over PyMC when sampling speed becomes the bottleneck, and **Bambi** when a formula interface
saves boilerplate.

## 8. Resources

- **Official docs** — https://www.pymc.io/
- **Example gallery** (canonical worked models) — https://www.pymc.io/projects/examples/
- **ArviZ docs** (diagnostics & plots for the `InferenceData` you get back) — https://python.arviz.org/
- **Statistical Rethinking** (the best on-ramp to applied Bayesian modeling; PyMC ports exist) — https://github.com/pymc-devs/pymc-resources
- **Bambi** (high-level formula interface built on PyMC) — https://bambinos.github.io/bambi/
- **"Why I use PyMC" / Towards a Principled Bayesian Workflow** — https://www.pymc.io/projects/docs/en/stable/learn.html

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def diagnose(chains, prob=0.89):
    """The between-versus-within-chain convergence statistic, and the pooled summary."""
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE